# Tune `rf_k_knn`

RF top-K + KNN. Repeated stratified CV on the train split; 
writes [`data/processed/tuned/rf_k_knn.json`](../data/processed/tuned/rf_k_knn.json).

**Stage 1 (hyperparameters):** SelectFromModel `max_features` (K), classifier `n_neighbors`. Select by **max mean PR AUC**.

**Stage 2 (threshold):** sweep `THRESHOLD_GRID` on the same CV folds; select threshold that **minimizes mean BER**.

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == "tuning" else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import scripts.tuning.registry as tuning_registry

importlib.reload(tuning_registry)
from scripts.secom_pipelines import TARGET_COL, feature_columns, load_mart, split_train_test
from scripts.tuning.registry import (
    MODEL_SPECS,
    fit_with_progress,
    run_grid_search,
    save_tuned_params,
    summarize_cv_search,
    tune_classifier_threshold,
    tuned_params_path,
)

MODEL_ID = "rf_k_knn"
spec = MODEL_SPECS[MODEL_ID]


In [2]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
print(len(X_train), "train rows", len(test_df), "test rows (holdout, not used here)")


1253 train rows 314 test rows (holdout, not used here)


In [3]:
param_grid = spec.make_param_grid()
pd.DataFrame([{k: v} for k, v in param_grid.items()])


,preprocess__sensor_mspc__select__max_features,classifier__n_neighbors
0,"[20, 30, 40, 50]",NaN
1,NaN,"[5, 10, 15]"


In [4]:
search, n_candidates, n_splits, total_fits = run_grid_search(spec, X_train, y_train)
print(f"{MODEL_ID}: {n_candidates} candidates x {n_splits} folds = {total_fits} fits")
search = fit_with_progress(search, X_train, y_train)


rf_k_knn: 12 candidates x 25 folds = 300 fits


GridSearchCV 300 fits:   0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

Fitting 25 folds for each of 12 candidates, totalling 300 fits


In [5]:
cv_summary, fold_results, aggregated = summarize_cv_search(search, spec)
print("Stage 1 best (mean PR AUC):")
display(aggregated.head(10))


Stage 1 best (mean PR AUC):


,top_k,n_neighbors,mean_ber_percent,std_ber_percent,mean_balanced_accuracy,mean_true_positive_percent,std_true_positive_percent,mean_true_negative_percent,std_true_negative_percent,mean_roc_auc,std_roc_auc,mean_pr_auc,std_pr_auc
7,40,10,49.758547,0.851338,0.502415,0.500000,1.695582,99.982906,0.083743,0.663589,0.049359,0.169729,0.052387
8,40,15,50.000000,0.000000,0.500000,0.000000,0.000000,100.000000,0.000000,0.676595,0.056721,0.165860,0.049415
2,20,15,50.008547,0.041872,0.499915,0.000000,0.000000,99.982906,0.083743,0.671206,0.055825,0.164342,0.050643
11,50,15,49.875000,0.612372,0.501250,0.250000,1.224745,100.000000,0.000000,0.679351,0.059344,0.162713,0.047260
5,30,15,49.757353,0.823267,0.502426,0.485294,1.646534,100.000000,0.000000,0.684806,0.049129,0.161876,0.041288
10,50,10,49.899447,0.582720,0.501006,0.235294,1.152701,99.965812,0.115937,0.667993,0.054056,0.154268,0.054760
4,30,10,49.782994,0.833686,0.502170,0.485294,1.646534,99.948718,0.138872,0.656239,0.040354,0.148123,0.038479
9,50,5,48.099045,2.530507,0.519010,4.588235,5.207751,99.213675,0.549673,0.649975,0.057018,0.140141,0.046470
1,20,10,49.774447,0.830316,0.502256,0.485294,1.646534,99.965812,0.115937,0.655503,0.059140,0.137652,0.039958
6,40,5,48.132416,1.746456,0.518676,4.367647,3.651859,99.367521,0.420805,0.634093,0.047793,0.132110,0.028599


In [6]:
threshold_result = tune_classifier_threshold(spec, X_train, y_train, cv_summary)
print(f"Stage 2 best threshold: {threshold_result['best_threshold']:.2f}")
print(f"  mean BER at threshold: {threshold_result['mean_ber_percent']:.2f}%")
display(threshold_result["per_threshold_mean_ber"].head(10))


Threshold CV folds:   0%|          | 0/25 [00:00<?, ?it/s]

Stage 2 best threshold: 0.05
  mean BER at threshold: 36.23%


,threshold,mean_ber_percent
0,0.05,36.231209
1,0.10,36.231209
2,0.15,38.806498
3,0.20,38.806498
4,0.25,43.998555
5,0.30,43.998555
6,0.35,45.984791
7,0.40,45.984791
8,0.45,48.285194
9,0.50,48.285194


In [7]:
payload = save_tuned_params(
    spec,
    cv_summary,
    fold_results,
    aggregated,
    threshold_result=threshold_result,
)
out_path = tuned_params_path(MODEL_ID)
print(f"Wrote {out_path}")
payload["grid_search_best_params"]


Wrote /home/troy/SECOM/data/processed/tuned/rf_k_knn.json


{'preprocess__sensor_mspc__select__max_features': 40,
 'classifier__n_neighbors': 10}